In [1]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
import time

In [2]:
CSV_PATH = r"C:\Users\USER\OneDrive\바탕 화면\Ckrasseck\Study\주전공\3-1\동양중세사\소논문\데이터\kangxi_master_A_1661_1722.csv"   # ← 파일 경로 수정
CHUNK_SIZE = 50_000             # ← 메모리 상황에 따라 조절 (행 단위)


chunks = []
start = time.time()

for i, chunk in enumerate(pd.read_csv(CSV_PATH, chunksize=CHUNK_SIZE)):
    # ✅ Null → 0 처리
    chunk = chunk.fillna(0)
    chunks.append(chunk)
    print(f"  청크 {i+1} 로드 완료 | 행: {len(chunk)}")

df = pd.concat(chunks, ignore_index=True)

print(f"\n✅ 전체 로딩 완료")
print(f"   Shape : {df.shape}")
print(f"   소요시간 : {time.time() - start:.2f}초")
print(f"   메모리 사용량 : {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

  청크 1 로드 완료 | 행: 35171

✅ 전체 로딩 완료
   Shape : (35171, 63)
   소요시간 : 0.51초
   메모리 사용량 : 19.0 MB


In [3]:
import pandas as pd

print(df.value_counts())

한자단어  1661  1662  1663  1664  1665  1666  1667  1668  1669  1670  1671  1672  1673  1674  1675  1676  1677  1678  1679  1680  1681  1682  1683  1684  1685  1686  1687  1688  1689  1690  1691  1692  1693  1694  1695  1696  1697  1698  1699  1700  1701  1702  1703  1704  1705  1706  1707  1708  1709  1710  1711  1712  1713  1714  1715  1716  1717  1718  1719  1720  1721  1722
等     28.0  11.0  16.0  29.0  18.0  14.0  14.0  16.0  43.0  11.0  13.0  13.0  14.0  55.0  52.0  44.0  39.0  64.0  44.0  49.0  34.0  34.0  36.0  23.0  44.0  26.0  17.0  29.0  18.0  36.0  57.0  32.0  30.0  28.0  34.0  60.0  61.0  33.0  31.0  37.0  24.0  18.0  20.0  27.0  17.0  25.0  22.0  26.0  20.0  21.0  25.0  22.0  37.0  32.0  46.0  36.0  52.0  45.0  31.0  29.0  33.0  25.0    1
皇帝    14.0  10.0  3.0   4.0   2.0   3.0   3.0   3.0   4.0   8.0   2.0   2.0   5.0   7.0   5.0   5.0   7.0   7.0   7.0   5.0   5.0   7.0   4.0   5.0   4.0   4.0   2.0   4.0   4.0   5.0   8.0   7.0   6.0   8.0   8.0   16.0  21.0  8.0   10.0  7

In [4]:
# ── Case A ──────────────────────────────────────────
# object 컬럼을 인덱스로 빼고, 나머지 숫자만 희소행렬로

# object 컬럼과 숫자 컬럼 분리
obj_cols  = df.select_dtypes(include='object').columns.tolist()
num_cols  = df.select_dtypes(exclude='object').columns.tolist()

# 라벨은 따로 저장
label_df  = df[obj_cols].copy()
num_df    = df[num_cols].copy()

print(f"라벨 컬럼 : {obj_cols}")
print(f"숫자 컬럼 수 : {len(num_cols)}")
print(label_df.head(3))

# ✅ 숫자 컬럼만 희소행렬 변환
sparse_matrix = csr_matrix(num_df.values.astype(np.float32))

print(f"\n✅ 희소행렬 변환 완료 | Shape: {sparse_matrix.shape}")
print(f"   Non-zero 원소: {sparse_matrix.nnz:,}")

# DataFrame → numpy → CSR 희소행렬
sparse_matrix = csr_matrix(num_df.values.astype(np.float32))

print(f"✅ 희소행렬 변환 완료")
print(f"   Shape  : {sparse_matrix.shape}")
print(f"   저장된 원소 수 (non-zero) : {sparse_matrix.nnz:,}")
print(f"   희소율 (sparsity) : {1 - sparse_matrix.nnz / (sparse_matrix.shape[0] * sparse_matrix.shape[1]):.2%}")


라벨 컬럼 : ['한자단어']
숫자 컬럼 수 : 62
  한자단어
0    等
1   皇帝
2   大臣

✅ 희소행렬 변환 완료 | Shape: (35171, 62)
   Non-zero 원소: 82,840
✅ 희소행렬 변환 완료
   Shape  : (35171, 62)
   저장된 원소 수 (non-zero) : 82,840
   희소율 (sparsity) : 96.20%


C:\Users\USER\AppData\Local\Temp\ipykernel_16724\3597664452.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  obj_cols  = df.select_dtypes(include='object').columns.tolist()


In [5]:
# 인덱스 세팅
num_df.index = label_df['한자단어'].values

# ① '위기' 행 → y (타겟)
y_row = num_df.loc['위기']                     # shape: (n_samples,)

# ② 나머지 35,169개 행 → X (피처)
X_df = num_df.drop(index='위기')               # shape: (35169, n_samples)

# ③ 전치: 행/열 뒤집기
X = X_df.T                                     # shape: (n_samples, 35169)
y_raw = y_row.values                           # shape: (n_samples,)

print(f"X shape : {X.shape}   → (샘플 수, 단어 수)")
print(f"y shape : {y_raw.shape}")
print(f"y 값 샘플 : {y_raw[:10]}")

# y값이 연속형(빈도/횟수)이면 0/1로 변환
y = (y_raw > 0).astype(int)

print(f"y=0 (위기 없음) : {(y==0).sum():,}개")
print(f"y=1 (위기 있음) : {(y==1).sum():,}개")


X shape : (62, 35170)   → (샘플 수, 단어 수)
y shape : (62,)
y 값 샘플 : [2. 2. 0. 0. 0. 0. 2. 0. 4. 0.]
y=0 (위기 없음) : 34개
y=1 (위기 있음) : 28개


In [6]:
import pandas as pd

df = pd.DataFrame(X)
df['y'] = y

counts = df['y'].value_counts()
valid_classes = counts[counts >= 2].index

df = df[df['y'].isin(valid_classes)]

X_new = df.drop(columns='y')
y_new = df['y']

In [7]:
from scipy.sparse import csr_matrix
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# ── 1. X, y 세팅 ──────────────────────────────────
num_df.index = label_df['한자단어'].values

y = num_df.loc['위기'].values.astype(int)          # 이진 (0/1)
X_df = num_df.drop(index='위기').T                 # (샘플 수, 35169)
X = csr_matrix(X_df.values.astype(np.float32))

print(f"X shape : {X.shape}")
print(f"y shape : {y.shape}")
print(f"y=0 : {(y==0).sum():,}개  |  y=1 : {(y==1).sum():,}개")

# ── 2. Train / Test 분할 ──────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X_new, y_new, test_size=0.2, stratify=y_new, random_state=42
)
print(f"\nTrain: {X_train.shape}  |  Test: {X_test.shape}")

# ── 3. 로지스틱 학습 ──────────────────────────────
model = LogisticRegression(
    solver='saga',
    max_iter=1000,
    class_weight='balanced',   # ✅ 0/1 불균형 자동 보정
    n_jobs=-1,
    verbose=1
)
model.fit(X_train, y_train)

# ── 4. 평가 ───────────────────────────────────────
y_pred = model.predict(X_test)

print(f"\n정확도 : {accuracy_score(y_test, y_pred):.4f}")
print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred, target_names=['위기없음(0)', '위기있음(1)']))
print("=== Confusion Matrix ===")
print(confusion_matrix(y_test, y_pred))


X shape : (62, 35170)
y shape : (62,)
y=0 : 34개  |  y=1 : 13개

Train: (49, 35170)  |  Test: (13, 35170)


c:\Users\USER\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


max_iter reached after 20 seconds

정확도 : 0.8462

=== Classification Report ===
              precision    recall  f1-score   support

     위기없음(0)       0.86      0.86      0.86         7
     위기있음(1)       0.83      0.83      0.83         6

    accuracy                           0.85        13
   macro avg       0.85      0.85      0.85        13
weighted avg       0.85      0.85      0.85        13

=== Confusion Matrix ===
[[6 1]
 [1 5]]


c:\Users\USER\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

model_lasso = LogisticRegression(
    penalty='l1',              # ✅ Lasso (L1 규제)
    solver='saga',             # L1 + 희소행렬 지원하는 유일한 solver
    C=1.0,                     # 규제 강도 (작을수록 강한 규제)
    class_weight='balanced',   # 클래스 불균형 보정
    max_iter=1000,
    n_jobs=-1,
    verbose=1
)

model_lasso.fit(X_train, y_train)
y_pred = model_lasso.predict(X_test)

print(f"\n=== Classification Report ===")
print(f"\n정확도 : {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred,
      target_names=['위기없음(0)', '위기있음(1)']))
print(f"=== Confusion Matrix ===")
print(confusion_matrix(y_test, y_pred))

import pandas as pd
import numpy as np

# 계수가 0이 아닌 단어 = Lasso가 선택한 단어
coef = model_lasso.coef_[0]
word_names = X_df.columns.tolist()   # 한자단어 35169개

importance_df = pd.DataFrame({
    '한자단어': word_names,
    '계수': coef
})

# 0이 아닌 것만 필터링
selected = importance_df[importance_df['계수'] != 0]
selected = selected.reindex(selected['계수'].abs().sort_values(ascending=False).index)

print(f"전체 단어 수     : {len(word_names):,}개")
print(f"Lasso 선택 단어  : {len(selected):,}개")
print(f"\n=== 위기 예측에 중요한 상위 20개 한자 ===")
print(selected.head(20).to_string(index=False))


c:\Users\USER\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\USER\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
c:\Users\USER\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


max_iter reached after 28 seconds

=== Classification Report ===

정확도 : 0.7692
              precision    recall  f1-score   support

     위기없음(0)       0.83      0.71      0.77         7
     위기있음(1)       0.71      0.83      0.77         6

    accuracy                           0.77        13
   macro avg       0.77      0.77      0.77        13
weighted avg       0.78      0.77      0.77        13

=== Confusion Matrix ===
[[5 2]
 [1 5]]


c:\Users\USER\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


전체 단어 수     : 35,170개
Lasso 선택 단어  : 188개

=== 위기 예측에 중요한 상위 20개 한자 ===
한자단어        계수
   曰  0.211080
 按察使 -0.172807
  喇嘛  0.138454
   係  0.130831
  蒙古 -0.120322
   矣 -0.106481
   祭 -0.100065
   軍  0.098902
 皇太后 -0.093112
  吏部  0.087616
   米 -0.083719
 布政使 -0.077249
  盛京 -0.077241
   內  0.075987
   無  0.075170
  地方 -0.072943
  達賴  0.072534
   寺  0.071539
  西安  0.070807
   或  0.070044


In [9]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f"SMOTE 전 → y=0: {(y_train==0).sum():,}  y=1: {(y_train==1).sum():,}")
print(f"SMOTE 후 → y=0: {(y_train_sm==0).sum():,}  y=1: {(y_train_sm==1).sum():,}")

model_sm = LogisticRegression(
    penalty='l1', solver='saga', C=1.0,
    max_iter=1000, n_jobs=-1, verbose=1
)
model_sm.fit(X_train_sm, y_train_sm)

print(classification_report(y_test, model_sm.predict(X_test),
      target_names=['위기없음(0)', '위기있음(1)']))

SMOTE 전 → y=0: 27  y=1: 22
SMOTE 후 → y=0: 27  y=1: 27


c:\Users\USER\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\USER\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
c:\Users\USER\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


max_iter reached after 30 seconds
              precision    recall  f1-score   support

     위기없음(0)       0.83      0.71      0.77         7
     위기있음(1)       0.71      0.83      0.77         6

    accuracy                           0.77        13
   macro avg       0.77      0.77      0.77        13
weighted avg       0.78      0.77      0.77        13



c:\Users\USER\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
